<a href="https://colab.research.google.com/github/ljcheng84/JC-NUS-ISS-ARCHAAS-STE-2026/blob/main/IT_Operations_L1_2_3_Triage_Flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#IT Service Desk AI Triage Simulation

##Overview

This project implements an automated, multi-agent IT Service Desk triage system using the CrewAI framework. The goal of this simulation is to intelligently process incoming user IT support tickets, categorize their domain, and route them to specialized engineering agents for immediate, actionable resolution.

To simulate a highly efficient zero-touch support environment, this system enforces strict "no-question" constraints. Agents are explicitly forbidden from asking users clarifying questions or requesting diagnostic logs. Instead, they are instructed to make enterprise-standard assumptions based on the provided prompt and immediately output actionable troubleshooting commands and remediation steps.

##Agent Architecture

The simulation utilizes a hierarchical L1-to-L3 routing structure:

L1 Service Desk Triage: Acts as the intake router. Analyzes the user prompt and delegates to specific L2 domains or escalates to L3.

L2 Domain Engineers (Network, System, Security): Specialized agents that handle purely single-domain issues using strict domain expertise.

L3 Technical Manager: Handles complex, cross-domain incidents by dissecting the user prompt, coordinating sub-tasks with the L2 agents, and synthesizing a unified final resolution.

##Setup Instructions

This project is designed to be run sequentially within a Google Colab notebook environment.

##Prerequisites

A Google Account to access Google Colab.

An active OpenAI API key.

#Step 1: Install Dependencies

Run the following pip install command in your first Colab cell to ensure the CrewAI framework is available in your runtime environment.

%pip install -q crewai nest_asyncio


#Step 2: Configure API Keys

To securely load your OpenAI API key, use Google Colab's built-in Secrets manager (the key icon on the left sidebar). Add a new secret named OPENAI_API_KEY.

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


#Step 3: Define Models, Agents, and Workflow

Run the cells containing your specific project code:

Define the LLMs: Instantiate the LLM objects (e.g., gpt-4o-mini for L1/L2, gpt-4o for L3) with low temperatures (e.g., 0.2) for deterministic outputs.

Define the Agents: Execute the cell containing the definitions for l1_service_desk, l2_security, l2_system, l2_network, and l3_manager. Ensure allow_delegation=True is enabled.

Define the Processing Logic: Execute the process_it_ticket(user_prompt) function containing the Task and Crew workflow definitions.

#Step 4: Run the Simulation (Mock data / Manual Input)

To interact with the simulation, run the following code cell. It will generate a text box where you can manually inject IT tickets. The system will process the text, route it to the proper agent, and output the assumed resolution.


# Step 1 - Installations & Environment Setup

In [26]:
%pip install -q crewai

import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM

# Load API Key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Step 2 - Define Delegation Logic and Execution Loop

In [27]:
async def process_it_ticket(user_prompt: str):
    print(f"\n--- Processing New Ticket: {user_prompt} ---")

    # Define the intake task dynamically based on user input
    triage_task = Task(
        description=f"""
        A user has submitted the following IT issue: "{user_prompt}"

        Instructions:
        1. Read the ticket and identify the IT domains involved.
        2. Consult your ROUTING DECISION TREE.
        3. Make EXACTLY ONE tool call to "Delegate work to co-worker" to assign the ticket to the correct agent.
        4. Return the exact output provided by the coworker as your Final Answer.
        """,
        expected_output="The final troubleshooting resolution generated by the delegated agent.",
        agent=l1_service_desk
    )

    # Assemble the crew
    it_support_crew = Crew(
        agents=[l1_service_desk, l2_security, l2_system, l2_network, l3_techmanager],
        tasks=[triage_task],
        process=Process.sequential, # L1 starts the sequence and delegates as needed
        verbose=False #hide the CrewOutput at the end, Agents has verbose logging which will show the agents interaction
    )

    # Execute the workflow asynchronously
    result = await it_support_crew.kickoff_async()
    return result


from crewai.tools import tool
from textwrap import dedent

@tool("L2 Delegation Tool")
def l2_delegation_tool(domain: str, task_description: str) -> str:
    """
    Use this tool to delegate tasks to L2 Engineers.
    Provide the 'domain' (either 'Network' or 'System') and the 'task_description'.
    This tool will automatically contact the correct L2 Engineer and return their response.
    """
    if domain.lower() == "network":
        # Programmatically execute the L2 Network task
        from crewai import Task
        temp_task = Task(
            description=task_description,
            expected_output="Actionable troubleshooting steps.",
            agent=l2_network
        )
        return temp_task.execute_sync()

    elif domain.lower() == "system":
        # Programmatically execute the L2 System task
        from crewai import Task
        temp_task = Task(
            description=task_description,
            expected_output="Actionable troubleshooting steps.",
            agent=l2_system
        )
        return temp_task.execute_sync()

    return "Error: Invalid domain. Must be 'Network' or 'System'."

# Step 3 - Agents Definition

In [28]:
# Define the tiered LLMs
l1_llm = LLM(model="gpt-4o-mini", temperature=0.1)
l2_llm = LLM(model="gpt-4o-mini", temperature=0.5)
l3_llm = LLM(model="gpt-4o", temperature=0.7)

# 1. L1 Service Desk Agent
l1_service_desk = Agent(
    role='L1 Service Desk Triage',
    goal='Categorize incoming IT tickets and trigger exactly ONE delegation.',
    backstory="""You are the first point of contact for IT issues. Your ONLY job is to categorize the ticket and use the "Delegate work to co-worker" tool to route it.

ROUTING DECISION TREE (Follow strictly):
- IF the issue is purely Network (e.g., WAN, BGP, routing, internet), YOU MUST delegate to "L2 Network Engineer".
- IF the issue is purely System (e.g., Active Directory, Windows OS, servers, reboots), YOU MUST delegate to "L2 System Engineer".
- IF the issue is purely Security (e.g., firewall patching, IAM, access), YOU MUST delegate to "L2 Security Engineer".
- IF AND ONLY IF the issue spans multiple distinct domains (e.g., a database crash AND a firewall drop), YOU MUST delegate to "L3 Technical Manager".

CRITICAL RULE: Do not default to the L3 Technical Manager for single-domain issues. You must use the tool EXACTLY ONCE per ticket.""",
    verbose=True,
    allow_delegation=True,
    llm=l1_llm
)

# 2. L2 Security Engineer
l2_security = Agent(
    role='L2 Security Engineer',
    goal='Resolve security-related IT tickets including firewall rules, compromised accounts, and access management immediately.',
    backstory='You are a cybersecurity specialist. Do NOT ask the user for follow-up details, error logs, or clarifying questions. You must work exclusively with the information provided. Provide an initial, actionable set of recommended troubleshooting steps immediately and return the final troubleshooting steps directly to the agent who assigned it to you.. If critical diagnostic information is missing, explicitly state your assumptions (e.g., "Assuming standard Microsoft Entra ID integration") and provide the resolution based on those enterprise baselines.',
    verbose=True,
    allow_delegation=False, #L2 cannot reassign task to other people
    llm=l2_llm
)

# 3. L2 System Engineer
l2_system = Agent(
    role='L2 System Engineer',
    goal='Resolve OS, server, Active Directory, and application-level IT tickets immediately.',
    backstory='You are an enterprise systems administrator. Do NOT ask the user for follow-up details, error logs, or clarifying questions. You must work exclusively with the information provided. Provide an initial, actionable set of recommended troubleshooting steps immediately and and return the final troubleshooting steps directly to the agent who assigned it to you.. If critical diagnostic information is missing, explicitly state your assumptions (e.g., "Assuming Windows Server 2022 and standard AD DS") and provide the resolution based on those enterprise baselines.',
    verbose=True,
    allow_delegation=False, #L2 cannot reassign task to other people
    llm=l2_llm
)

# 4. L2 Network Engineer
l2_network = Agent(
    role='L2 Network Engineer',
    goal='Resolve routing, switching, VPN, and general connectivity IT tickets immediately.',
    backstory='You are a core networking expert. Do NOT ask the user for follow-up details, IP addresses, or clarifying questions. You must work exclusively with the information provided. Provide an initial, actionable set of recommended troubleshooting steps immediately and return the final troubleshooting steps directly to the agent who assigned it to you.. If critical diagnostic information is missing, explicitly state your assumptions (e.g., "Assuming Cisco core routing and standard IPSec VPN") and provide the resolution based on those enterprise baselines.',
    verbose=True,
    allow_delegation=False, #L2 cannot reassign task to other people
    llm=l2_llm
)

# 5. L3 Technical Manager
l3_techmanager = Agent(
    role='L3 Technical Manager',
    goal='Resolve cross-domain issues by actively delegating to L2 engineers and synthesizing their results.',
    backstory=f"""'You handle high-severity cross-domain incidents. You MUST use the "Delegate work to co-worker" tool to assign tasks to the L2 Network Engineer and L2 System Engineer.
    CRITICAL RULES:
    1. Explain your plan and engineers to delegate. The very first backend action MUST be using the delegation tool.
    2. You are STRICTLY FORBIDDEN from providing a "Final Answer" until you have received the actual troubleshooting steps back from both L2 engineers.
    3. Do not output raw JSON or bracketed text. Use the tool natively.',
    """,
    verbose=True,
    allow_delegation=False, # <-- Turn off native delegation
    tools=[l2_delegation_tool], # <-- Assign the custom tool
    llm=l3_llm
)

# Step 4a - Mock Task Assignment

In [29]:
# Test 1: Route to L2 Network
await process_it_ticket("Our branch office in Jurong has completely lost internet connectivity. The local switch is powered on, but the primary WAN link interface is showing continuous packet loss and BGP neighbor adjacency is down.")

# Test 2: Route to L2 System
await process_it_ticket("The primary Active Directory domain controller is stuck in a continuous reboot loop after the overnight Windows Server OS patching cycle, preventing users from logging into their local workstations.")

# Test 3: Route to L2 Security
await process_it_ticket("There is a firewall issue during firewall patching failover activities. We need immediate remediation.")

# Test 4: Route to L3 for Cross-Domain dissecting
await process_it_ticket("Our main customer-facing database server crashed with an out-of-memory error, while simultaneously our edge firewall started dropping all inbound traffic due to an alert, causing the network VPN gateway to become unreachable.")


--- Processing New Ticket: Our branch office in Jurong has completely lost internet connectivity. The local switch is powered on, but the primary WAN link interface is showing continuous packet loss and BGP neighbor adjacency is down. ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L1 Service Desk Triage                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│          A user has submitted the following IT issue: "Our branch office in Jurong has completely lost          │
│  internet connectivity. The local switch is powered on, but the primary WAN link interface is showing           │
│  continuous packet loss and BGP neighbor adjacency is down."                                                    │
│                                                                                                                 │
│          Instructions:                                                                                          │
│          1. Read the ticket and identify the IT domains involved.                                               │
│          2. Consult your ROUTING DECISION TREE.                                                                 │
│          3. Make EXACTLY ONE tool call to "Delegate work to co-worker" to assign the ticket to the correct      │
│  agent.                                                                                                         │
│          4. Return the exact output provided by the coworker as your Final Answer.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L2 Network Engineer                                                                                     │
│                                                                                                                 │
│  Task: Investigate and resolve the internet connectivity issue at the Jurong branch office. The local switch    │
│  is powered on, but the primary WAN link interface is showing continuous packet loss and BGP neighbor           │
│  adjacency is down. Please check the WAN link and BGP configurations to restore connectivity.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L2 Network Engineer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. **Check Physical Connectivity**:                                                                            │
│     - Verify that the physical WAN link (cables, connectors) is securely connected on both ends (local router   │
│  and ISP demarcation point).                                                                                    │
│     - Ensure that the WAN interface is not administratively down. Use the command `show ip interface brief` to  │
│  check the status.                                                                                              │
│                                                                                                                 │
│  2. **Examine WAN Link Status**:                                                                                │
│     - Run diagnostics on the WAN interface. Use commands like `ping` and `traceroute` to test connectivity to   │
│  the ISP and check for packet loss.                                                                             │
│     - Check for any errors or drops on the WAN interface using the command `show interfaces [interface-name]`.  │
│                                                                                                                 │
│  3. **Check BGP Configuration**:                                                                                │
│     - Use the command `show ip bgp summary` to confirm the status of BGP neighbors and identify if the          │
│  neighbor is down due to incorrect configuration or network issues.                                             │
│     - Verify the BGP configuration on the router. Ensure that the BGP neighbor IP addresses are correctly       │
│  configured and that the correct AS numbers are being used.                                                     │
│                                                                                                                 │
│  4. **Review Routing Table**:                                                                                   │
│     - Use the command `show ip route` to check if the routes to the internet are being learned correctly. Look  │
│  for any missing routes that could indicate a BGP issue.                                                        │
│                                                                                                                 │
│  5. **Check for Access Control Lists (ACLs)**:                                                                  │
│     - Ensure that no ACLs are blocking BGP traffic (TCP port 179) or any other necessary traffic on the WAN     │
│  interface. Use `show access-lists` to review ACLs.                                                             │
│                                                                                                                 │
│  6. **Examine Logs for Errors**:                                                                                │
│     - Review the router logs using the command `show logging` to look for any error messages related to the     │
│  WAN interface or BGP.                                                                                          │
│                                                                                                                 │
│  7. **Test with a Different WAN Link**:                

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L3 Technical Manager                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The tasks have been delegated to the L2 engineers, and here are the detailed troubleshooting steps they have   │
│  provided for both the database and network issues:                                                             │
│                                                                                                                 │
│  ### Database Server Out-of-Memory Error                                                                        │
│                                                                                                                 │
│  1. **Access the Database Server**: Log in to the database server using administrative credentials.             │
│                                                                                                                 │
│  2. **Locate the Log Files**: Navigate to the directory where the database server logs are stored. Common       │
│  locations include:                                                                                             │
│     - For SQL Server: `C:\Program Files\Microsoft SQL Server\MSSQL<version>.MSSQLSERVER\MSSQL\Log\`             │
│     - For MySQL: `/var/log/mysql/` or `/var/lib/mysql/`                                                         │
│     - For Oracle: `$ORACLE_HOME/diag/rdbms/<dbname>/<instance>/trace/`                                          │
│                                                                                                                 │
│  3. **Check Memory Usage**: Open the relevant log file (e.g., `error.log`, `sqlserver.log`, `mysql.log`) and    │
│  search for entries related to memory usage. Look for messages indicating memory allocation failures or         │
│  out-of-memory errors.                                                                                          │
│                                                                                                                 │
│  4. **Analyze Memory Settings**: Review the database server’s configuration settings for memory allocation:     │
│     - For SQL Server, check the maximum server memory setting in SQL Server Management Studio (SSMS) under      │
│  server properties.                                                                                             │
│     - For MySQL, check the `innodb_buffer_pool_size` and other memory-related settings in the `my.cnf` or       │
│  `my.ini` file.                                                                                                 │
│     - For Oracle, review the `SGA` and `PGA` settings.                                                          │
│                                                                                                                 │
│  5. **Check System Memory**: Use system monitoring tools to check overall memory usage on the server:           │
│     - On Windows, use Task Manager or Resource Monitor.                                                         │
│     - On Linux, use commands like `free -m`, `top`, or `htop`.                                                  │
│                                                                                                                 │
│  6. **Investigate Running Queries**: Identify any long-running or resource-intensive queries that may be        │
│  consuming excessive memory. Use database-specific tool

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L1 Service Desk Triage                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The tasks have been delegated to the L2 engineers, and here are the detailed troubleshooting steps they have   │
│  provided for both the database and network issues:                                                             │
│                                                                                                                 │
│  ### Database Server Out-of-Memory Error                                                                        │
│                                                                                                                 │
│  1. **Access the Database Server**: Log in to the database server using administrative credentials.             │
│                                                                                                                 │
│  2. **Locate the Log Files**: Navigate to the directory where the database server logs are stored. Common       │
│  locations include:                                                                                             │
│     - For SQL Server: `C:\Program Files\Microsoft SQL Server\MSSQL<version>.MSSQLSERVER\MSSQL\Log\`             │
│     - For MySQL: `/var/log/mysql/` or `/var/lib/mysql/`                                                         │
│     - For Oracle: `$ORACLE_HOME/diag/rdbms/<dbname>/<instance>/trace/`                                          │
│                                                                                                                 │
│  3. **Check Memory Usage**: Open the relevant log file (e.g., `error.log`, `sqlserver.log`, `mysql.log`) and    │
│  search for entries related to memory usage. Look for messages indicating memory allocation failures or         │
│  out-of-memory errors.                                                                                          │
│                                                                                                                 │
│  4. **Analyze Memory Settings**: Review the database server’s configuration settings for memory allocation:     │
│     - For SQL Server, check the maximum server memory setting in SQL Server Management Studio (SSMS) under      │
│  server properties.                                                                                             │
│     - For MySQL, check the `innodb_buffer_pool_size` and other memory-related settings in the `my.cnf` or       │
│  `my.ini` file.                                                                                                 │
│     - For Oracle, review the `SGA` and `PGA` settings.                                                          │
│                                                                                                                 │
│  5. **Check System Memory**: Use system monitoring tools to check overall memory usage on the server:           │
│     - On Windows, use Task Manager or Resource Monitor.                                                         │
│     - On Linux, use commands like `free -m`, `top`, or `htop`.                                                  │
│                                                                                                                 │
│  6. **Investigate Running Queries**: Identify any long-running or resource-intensive queries that may be        │
│  consuming excessive memory. Use database-specific tool

CrewOutput(raw="The tasks have been delegated to the L2 engineers, and here are the detailed troubleshooting steps they have provided for both the database and network issues:\n\n### Database Server Out-of-Memory Error\n\n1. **Access the Database Server**: Log in to the database server using administrative credentials.\n\n2. **Locate the Log Files**: Navigate to the directory where the database server logs are stored. Common locations include:\n   - For SQL Server: `C:\\Program Files\\Microsoft SQL Server\\MSSQL<version>.MSSQLSERVER\\MSSQL\\Log\\`\n   - For MySQL: `/var/log/mysql/` or `/var/lib/mysql/`\n   - For Oracle: `$ORACLE_HOME/diag/rdbms/<dbname>/<instance>/trace/`\n\n3. **Check Memory Usage**: Open the relevant log file (e.g., `error.log`, `sqlserver.log`, `mysql.log`) and search for entries related to memory usage. Look for messages indicating memory allocation failures or out-of-memory errors.\n\n4. **Analyze Memory Settings**: Review the database server’s configuration setti

# Step 4b - Manual user input and execution loop

In [30]:
# Prompt the user for manual input
manual_ticket = input("Enter the IT issue/ticket description: ")

# Feed the input directly into the execution function
if manual_ticket.strip():
    result = await process_it_ticket(manual_ticket)
    print("\n=== FINAL RESOLUTION ===")
    print(result)
else:
    print("No input provided. Please run the cell again and enter a ticket description.")

Enter the IT issue/ticket description: i encounter issues accessing the servers after changing GPO policies during patching activites

--- Processing New Ticket: i encounter issues accessing the servers after changing GPO policies during patching activites ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L1 Service Desk Triage                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│          A user has submitted the following IT issue: "i encounter issues accessing the servers after changing  │
│  GPO policies during patching activites"                                                                        │
│                                                                                                                 │
│          Instructions:                                                                                          │
│          1. Read the ticket and identify the IT domains involved.                                               │
│          2. Consult your ROUTING DECISION TREE.                                                                 │
│          3. Make EXACTLY ONE tool call to "Delegate work to co-worker" to assign the ticket to the correct      │
│  agent.                                                                                                         │
│          4. Return the exact output provided by the coworker as your Final Answer.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L2 System Engineer                                                                                      │
│                                                                                                                 │
│  Task: Investigate the issues accessing the servers that occurred after changing GPO policies during patching   │
│  activities.                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L2 System Engineer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To address the issues accessing the servers that occurred after changing GPO policies during patching          │
│  activities, please follow these troubleshooting steps:                                                         │
│                                                                                                                 │
│  1. **Review Recent GPO Changes**: Check the specific GPOs that were modified during the patching activities.   │
│  Identify any settings related to security, user permissions, and access control that may have been altered.    │
│                                                                                                                 │
│  2. **Use the Group Policy Results Tool**: Run the Group Policy Results (gpresult) command on the affected      │
│  servers to see which GPOs are currently applied and whether there are any denied permissions or unexpected     │
│  settings.                                                                                                      │
│                                                                                                                 │
│  3. **Check Event Viewer Logs**: Examine the Event Viewer on the affected servers for any warnings or errors    │
│  related to Group Policy processing. Look specifically for events under the "System" and "Application" logs     │
│  that occurred around the time of the patching activities.                                                      │
│                                                                                                                 │
│  4. **Verify User Permissions**: Ensure that the user experiencing access issues has the correct permissions    │
│  to access the servers. Check both local and domain-level permissions to confirm they align with the intended   │
│  access rights.                                                                                                 │
│                                                                                                                 │
│  5. **Test Connectivity**: Attempt to access the servers using different accounts, including administrative     │
│  accounts, to determine if the issue is user-specific or a broader access issue.                                │
│                                                                                                                 │
│  6. **Force Group Policy Update**: Run the command `gpupdate /force` on the affected servers to refresh the     │
│  Group Policy settings. This can help to apply any changes made and resolve inconsistencies.                    │
│                                                                                                                 │
│  7. **Rollback GPO Changes**: If the issue persists, consider rolling back the recent GPO changes to a          │
│  previous state and testing access again. This can be done by restoring the GPO from backup or using the Group  │
│  Policy Management Console (GPMC).                                                                              │
│                                                                                                                 │
│  8. **Check Firewall and Network Settings**: Ensure that any firewall or network settings have not been         │
│  inadvertently changed during the patching process that

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: L1 Service Desk Triage                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To address the issues accessing the servers that occurred after changing GPO policies during patching          │
│  activities, please follow these troubleshooting steps:                                                         │
│                                                                                                                 │
│  1. **Review Recent GPO Changes**: Check the specific GPOs that were modified during the patching activities.   │
│  Identify any settings related to security, user permissions, and access control that may have been altered.    │
│                                                                                                                 │
│  2. **Use the Group Policy Results Tool**: Run the Group Policy Results (gpresult) command on the affected      │
│  servers to see which GPOs are currently applied and whether there are any denied permissions or unexpected     │
│  settings.                                                                                                      │
│                                                                                                                 │
│  3. **Check Event Viewer Logs**: Examine the Event Viewer on the affected servers for any warnings or errors    │
│  related to Group Policy processing. Look specifically for events under the "System" and "Application" logs     │
│  that occurred around the time of the patching activities.                                                      │
│                                                                                                                 │
│  4. **Verify User Permissions**: Ensure that the user experiencing access issues has the correct permissions    │
│  to access the servers. Check both local and domain-level permissions to confirm they align with the intended   │
│  access rights.                                                                                                 │
│                                                                                                                 │
│  5. **Test Connectivity**: Attempt to access the servers using different accounts, including administrative     │
│  accounts, to determine if the issue is user-specific or a broader access issue.                                │
│                                                                                                                 │
│  6. **Force Group Policy Update**: Run the command `gpupdate /force` on the affected servers to refresh the     │
│  Group Policy settings. This can help to apply any changes made and resolve inconsistencies.                    │
│                                                                                                                 │
│  7. **Rollback GPO Changes**: If the issue persists, consider rolling back the recent GPO changes to a          │
│  previous state and testing access again. This can be done by restoring the GPO from backup or using the Group  │
│  Policy Management Console (GPMC).                                                                              │
│                                                                                                                 │
│  8. **Check Firewall and Network Settings**: Ensure that any firewall or network settings have not been         │
│  inadvertently changed during the patching process that


=== FINAL RESOLUTION ===
To address the issues accessing the servers that occurred after changing GPO policies during patching activities, please follow these troubleshooting steps:

1. **Review Recent GPO Changes**: Check the specific GPOs that were modified during the patching activities. Identify any settings related to security, user permissions, and access control that may have been altered.

2. **Use the Group Policy Results Tool**: Run the Group Policy Results (gpresult) command on the affected servers to see which GPOs are currently applied and whether there are any denied permissions or unexpected settings.

3. **Check Event Viewer Logs**: Examine the Event Viewer on the affected servers for any warnings or errors related to Group Policy processing. Look specifically for events under the "System" and "Application" logs that occurred around the time of the patching activities.

4. **Verify User Permissions**: Ensure that the user experiencing access issues has the correct perm